In [1]:
# setting root at top

import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

# Base Model with 64x64 Image and 4 Conv Blocks

### Import libraries
* `torch`
* `datasets`, `transforms` from `torchvision` for data Loading.
* `DataLoader` from `torch.utils.data` for Batching and using data in model.
* `BaseModel` from `architectures.BaseModel.py`
* `matplotlib.pyplot` as `plt`
* `seaborn` as sns

In [2]:
# training and architecture
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from architectures.BaseModel import BaseModel

# plotter
import matplotlib.pyplot as plt
import seaborn as sns

### Load datasets for training
* load 64x64 images
* load from `../dataset/final-dataset/train`
* convert to tensor
* make batch_size = 32

In [3]:
transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ]
)

dataset = datasets.ImageFolder(
    root="../dataset/final-dataset/train",
    transform=transform
)

dataloader = DataLoader(
    dataset=dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

### Use Model for training
Initialize model using 
* `conv_layers = 4`, 
* `initial_output_channel = 8` 
* `initial_image_size = 64`

In [4]:
model = BaseModel(conv_layers=4, initial_output_channel=8, initial_image_size=64).to("cuda")

In [5]:
def train_model(model, dataloader, lr = 0.001):
    cost = 0
    for (x,y) in dataloader:
        # to cuda
        x = x.to("cuda", non_blocking = True)
        y = y.to("cuda", non_blocking = True)
        # forward pass
        y_pred = model(x)
        # calculate loss
        loss = model.calculate_loss(y_pred, y)
        # backward propagation
        loss_val = model.backward(loss, lr)
        cost += loss_val
    
    return cost/len(dataloader)
        
        

In [ ]:
costs = []

for i in range(50):
    cost = train_model(model,dataloader, 0.005)
    costs.append(cost)
    if (cost+1)%10 == 0:
        print(f"Epoch {i+1}=> Loss: {cost}")